# 因果现金预测校准：一小时内的一次预登记尝试

假设：旧现金预测有方向信息，但幅度失准。只使用严格在当年 1 月 1 日前成熟的 2013 年以来 OOF 预测和一日标签，拟合 `y=a+b*p`，当年系数冻结。只跑这一个方案，不新增特征、搜索参数或组合路径。筛选门禁在 [plan.json](plan.json) 冻结；任意门禁失败即结束该路线，最多运行到 2026-10-03 02:04:34 UTC。

输入是原生 Qlib provider、DatasetH 与已完成父 Recorder，先重新读取日历/特征/原预测模型，核对父输入的精确一致性。原 14 个 Ridge 预测头不重训。开发期 2016–2022、已复用 2023 年至 as_of，以及用户开盘窗口分别报告。年表与用户窗口只给点估计；仅两段汇总窗口运行固定 20/60 长度、2000 次、seed=20261003 的配对 stationary bootstrap。

事件是单次 SOXX→现金→SOXX 的开盘间费用诊断，不能当成可实现组合收益。全部历史数据已经复用，没有新的独立前向证据，不能晋升。

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().resolve()
while not (ROOT / "SPEC.md").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import qlib
from qlib.data import D
from qlib.workflow import R
from IPython.display import display, Markdown
from src.workflow.causal_cash_calibration import EXPERIMENT, read_plan, run, conclusion
CARD = ROOT / "research" / EXPERIMENT
plan, identity = read_plan(CARD)
qlib.init(provider_uri=str(ROOT / plan["source"]["provider_uri"]), region="us", kernels=1)
calendar = D.calendar(start_time=plan["source"]["history_start"], end_time=plan["source"]["as_of"], freq="day")
assert str(calendar[-1].date()) == plan["source"]["as_of"]
print({"provider_uri": plan["source"]["provider_uri"], "calendar_sessions": len(calendar), "plan_sha256": identity["plan_sha256"]})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[55028:MainThread](2026-10-03 09:17:10,706) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55028:MainThread](2026-10-03 09:17:10,708) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55028:MainThread](2026-10-03 09:17:10,708) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-worktrees/causal-cash-calibration-20261003/data/qlib/published_futu_affine_20260925_v1')}


{'provider_uri': 'data/qlib/published_futu_affine_20260925_v1', 'calendar_sessions': 4162, 'plan_sha256': 'f7c0ab297fddb951071e2adb58b2f151f46470daf8ec2e354b6c9573490ddaa1'}


## Qlib 训练与证据

`run` 使用原生 Qlib 数据与 DatasetH 的冻结输入读取器，逐年严格筛选成熟 OOF 行，完成唯一一次 OLS 校准。所有模型系数、训练日期、预测明细、原始 bootstrap 抽样统计保存于 Qlib Recorder。再次执行只读取且校验同一已完成 Recorder；部分完成或证据散列变化时拒绝自动重跑。

In [2]:
summary = run(ROOT, CARD)
print({key: summary[key] for key in ("recorder_id", "screening_passed", "failed_gate_count", "decision", "calibration_fits", "forecast_model_refits", "new_portfolio_paths", "completed_at_utc")})
with R.uri_context(summary["recorder_uri"]):
    recorder = R.get_recorder(experiment_name=EXPERIMENT, recorder_id=summary["recorder_id"])
    tables = recorder.load_object("analysis/tables.pkl")
    fits = recorder.load_object("models/calibrations.pkl")
assert len(fits) == 11
# Parent loader created native DatasetH; details and maturity evidence are in Recorder.
display(tables["calibration_fits"][["model_year", "training_rows", "intercept", "slope", "last_training_signal", "return_available_at_max"]])

[55028:MainThread](2026-10-03 09:17:10,750) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55028:MainThread](2026-10-03 09:17:10,751) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55028:MainThread](2026-10-03 09:17:10,751) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-worktrees/causal-cash-calibration-20261003/data/qlib/published_futu_affine_20260925_v1')}


[55028:MainThread](2026-10-03 09:17:10,815) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55028:MainThread](2026-10-03 09:17:10,815) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


...

[55028:MainThread](2026-10-03 09:17:14,523) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[55028:MainThread](2026-10-03 09:17:14,523) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[55028:MainThread](2026-10-03 09:17:21,413) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'recorder_id': 'cade082cf1454cb19aa95d3cacc83b32', 'screening_passed': False, 'failed_gate_count': 21, 'decision': 'abandon_cash_calibration_route', 'calibration_fits': 11, 'forecast_model_refits': 0, 'new_portfolio_paths': 0, 'completed_at_utc': '2026-10-03T01:17:21.404887+00:00'}


,model_year,training_rows,intercept,slope,last_training_signal,return_available_at_max
0,2016,754,-0.000710,0.171047,2015-12-29,2015-12-31T14:30:00+00:00
1,2017,1006,-0.000799,0.268325,2016-12-28,2016-12-30T14:30:00+00:00
2,2018,1257,-0.000885,0.171005,2017-12-27,2017-12-29T14:30:00+00:00
3,2019,1508,-0.000601,0.261072,2018-12-27,2018-12-31T14:30:00+00:00
4,2020,1760,-0.000741,0.310438,2019-12-27,2019-12-31T14:30:00+00:00
5,2021,2013,-0.000655,0.446630,2020-12-29,2020-12-31T14:30:00+00:00
6,2022,2265,-0.000731,0.430970,2021-12-29,2021-12-31T14:30:00+00:00
7,2023,2516,-0.000426,0.404047,2022-12-28,2022-12-30T14:30:00+00:00
8,2024,2766,-0.000596,0.389768,2023-12-27,2023-12-29T14:30:00+00:00
9,2025,3018,-0.000579,0.362871,2024-12-27,2024-12-31T14:30:00+00:00


In [3]:
display(tables["point_metrics"][["window", "comparator", "rows", "MSE_prediction", "MSE_prior_baseline", "paired_MSE_improvement_baseline_minus_prediction", "time_series_Pearson_IC"]])
display(tables["event_metrics"].loc[tables["event_metrics"].window_type.eq("pooled"), ["window", "forecast_role", "cost_bps", "event_count", "coverage", "mean_event_net_log_bp"]])
display(tables["screening_gates"])
print("所有逐年/用户窗口、抽样置信区间与散列在根目录小表或 Qlib Recorder；没有组合回测。")

,window,comparator,rows,MSE_prediction,MSE_prior_baseline,paired_MSE_improvement_baseline_minus_prediction,time_series_Pearson_IC
0,signal_2016,raw,252,0.000190,0.000189,-1.304279e-06,0.108408
1,signal_2016,prior_mean,252,0.000190,0.000191,1.413162e-06,0.108408
2,signal_2017,raw,251,0.000135,0.000143,7.468929e-06,-0.072774
3,signal_2017,prior_mean,251,0.000135,0.000134,-7.715806e-07,-0.072774
4,signal_2018,raw,251,0.000328,0.000324,-4.284505e-06,0.142627
5,signal_2018,prior_mean,251,0.000328,0.000328,4.206255e-07,0.142627
6,signal_2019,raw,252,0.000233,0.000232,-7.911556e-07,0.116526
7,signal_2019,prior_mean,252,0.000233,0.000234,1.931562e-06,0.116526
8,signal_2020,raw,253,0.000623,0.000619,-3.622299e-06,0.128160
9,signal_2020,prior_mean,253,0.000623,0.000630,6.729003e-06,0.128160


,window,forecast_role,cost_bps,event_count,coverage,mean_event_net_log_bp
66,development_2016_2022,calibrated,10.0,9,0.005108,-65.129654
67,development_2016_2022,calibrated,25.0,0,0.000000,NaN
68,development_2016_2022,raw,10.0,145,0.082293,-18.649672
69,development_2016_2022,raw,25.0,19,0.010783,-95.836817
70,development_2016_2022,prior_mean,10.0,0,0.000000,NaN
71,development_2016_2022,prior_mean,25.0,0,0.000000,NaN
72,reused_2023_asof,calibrated,10.0,18,0.019272,59.430815
73,reused_2023_asof,calibrated,25.0,0,0.000000,NaN
74,reused_2023_asof,raw,10.0,141,0.150964,-3.122493
75,reused_2023_asof,raw,25.0,25,0.026767,17.318347


,gate,observed,required,passed,comparator,window,mean_block,cost_bps
0,annual_MSE,3.000000e+00,>=6 of 7,False,raw,NaN,NaN,NaN
1,pooled_MSE,8.807626e-07,>0,True,raw,development_2016_2022,NaN,NaN
2,pooled_MSE_lower95,-2.897968e-06,">0, all draws defined",False,raw,development_2016_2022,20.0,NaN
3,pooled_MSE_lower95,-2.530040e-06,">0, all draws defined",False,raw,development_2016_2022,60.0,NaN
4,pooled_MSE,3.869945e-07,>0,True,raw,reused_2023_asof,NaN,NaN
5,pooled_MSE_lower95,-6.551846e-06,">0, all draws defined",False,raw,reused_2023_asof,20.0,NaN
6,pooled_MSE_lower95,-7.045840e-06,">0, all draws defined",False,raw,reused_2023_asof,60.0,NaN
7,annual_MSE,5.000000e+00,>=6 of 7,False,prior_mean,NaN,NaN,NaN
8,pooled_MSE,1.452261e-06,>0,True,prior_mean,development_2016_2022,NaN,NaN
9,pooled_MSE_lower95,-9.212796e-07,">0, all draws defined",False,prior_mean,development_2016_2022,20.0,NaN


所有逐年/用户窗口、抽样置信区间与散列在根目录小表或 Qlib Recorder；没有组合回测。


## 产物和结果边界

入口：[workflow.yaml](workflow.yaml)、[summary.json](summary.json)、[conclusion.md](conclusion.md)。小表：[calibration_fits.csv](calibration_fits.csv)、[point_metrics.csv](point_metrics.csv)、[event_metrics.csv](event_metrics.csv)、[bootstrap_summary.csv](bootstrap_summary.csv)、[screening_gates.csv](screening_gates.csv)。大预测与原始 bootstrap draws 只存 Recorder。

API 权威依据：[Qlib Recorder 文档](https://qlib.readthedocs.io/en/latest/component/recorder.html)、[Qlib Data 文档](https://qlib.readthedocs.io/en/latest/component/data.html)。筛选通过也仅代表可再考虑，不能通过投资策略晋升门。

In [4]:
# 中文结论由本次完成的 Recorder 小表同步生成，避免手改指标漂移。
display(Markdown(conclusion(CARD)))

# 因果现金预测校准：限时单次筛选

本次只检验一个预登记方案：每年 1 月 1 日，用自 2013 年以来严格在此前成熟的 OOF 现金预测与 H1 标签，拟合 y=a+b×p；全年冻结 a、b。2016–2026 共 11 次两参数 OLS，原 14 个预测头未重训。

结论：筛选未通过，立即放弃这条现金校准路线。失败门禁 21 项；不启动其他配方、参数网格或组合回测。

development_2016_2022：1762 个已成熟样本；校准 MSE=0.000368963804。
- 对 raw：基准 MSE=0.000369844566，基准减校准的 MSE 改善=8.80762552e-07。
- 对 prior_mean：基准 MSE=0.000370416065，基准减校准的 MSE 改善=1.45226092e-06。
- 单边成本 10 bp：孤立事件 9 次，条件平均净相对 log 收益 -65.1297 bp。
- 单边成本 25 bp：孤立事件 0 次，条件平均净相对 log 收益 未定义（无事件）。

reused_2023_asof：934 个已成熟样本；校准 MSE=0.000550976095。
- 对 raw：基准 MSE=0.000551363089，基准减校准的 MSE 改善=3.8699451e-07。
- 对 prior_mean：基准 MSE=0.000553710225，基准减校准的 MSE 改善=2.73413041e-06。
- 单边成本 10 bp：孤立事件 18 次，条件平均净相对 log 收益 59.4308 bp。
- 单边成本 25 bp：孤立事件 0 次，条件平均净相对 log 收益 未定义（无事件）。

开发期七年中，对 raw 的 MSE 改善年数为 3/7（要求至少 6/7）。
开发期七年中，对 prior_mean 的 MSE 改善年数为 5/7（要求至少 6/7）。

用户窗口按 2026-08-03 交易开盘至 2026-09-25 最后已观察开盘限定：38 个成熟观测，校准 MSE=0.000663134964；年表与双窗口差异见 point_metrics.csv，事件与区间见 event_metrics.csv / bootstrap_summary.csv。未新增行情状态分层。
年度校准斜率范围 0.171005 至 0.489849；每次训练日期、成熟时间上界、系数和散列见 calibration_fits.csv。

风险：全部历史窗口已被复用，不属于新的独立样本；原数据的 published affine 与 serverQFQ 差异及 2012 年 1 月次级报价不确定性保留。现金事件只是 SOXX→现金→SOXX 的一期开盘费用测试，不能连乘或当成组合收益；不存在新前向证据，不晋升、不改生产。

下一步：本轮到此结束，保留失败或通过的完整证据；不把筛选结果声明为稳定超额收益。
Qlib Recorder：cade082cf1454cb19aa95d3cacc83b32；数据源 data/qlib/published_futu_affine_20260925_v1，as_of=2026-09-25；完成于 2026-10-03T01:17:21.404887+00:00（截止 2026-10-03T02:04:34Z）。
